# INHALEX — Recomendación de aroma complementario con Apriori

Esta libreta reproduce la propuesta 1 con datos sintéticos compatibles con la colección `pedidos`. Cada fila del dataset final representa una compra completa: `tid` identifica la canasta y `items` contiene los slugs de los aromas adquiridos juntos.

Apriori es no supervisado: no existe variable Y. El resultado son reglas `aroma A → aroma B` acompañadas de soporte, confianza y lift.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

REPO_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'ml').is_dir() and (path / 'Server').is_dir()
)
EXPORT_DIR = REPO_ROOT / 'ml' / 'exports'
ARTIFACT_DIR = REPO_ROOT / 'ml' / 'artifacts'

## 1. Construcción reproducible del dataset

El primer script conserva únicamente eventos `purchase`, agrupa por pedido, elimina cantidades repetidas y transforma cada producto a su slug estable. El segundo ejecuta una implementación propia y auditable de Apriori.

In [2]:
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'ml' / 'src' / 'build_apriori_dataset.py')],
    cwd=REPO_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'ml' / 'src' / 'train_apriori.py')],
    cwd=REPO_ROOT,
    check=True,
)

CompletedProcess(args=['C:\\Users\\David\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', 'c:\\Users\\David\\Documents\\Proyecto-INHALEX\\ml\\src\\train_apriori.py'], returncode=0)

## 2. Dataset transaccional TID / Items

La columna `items` usa `|` como separador. La cantidad de unidades no se repite porque Apriori clásico analiza presencia o ausencia dentro de cada canasta.

In [3]:
transactions = pd.read_csv(
    EXPORT_DIR / 'dataset_apriori_transacciones.csv',
    encoding='utf-8-sig',
)
transactions.head(10)

,tid,items
0,SYN-ORD-000001,jengibre|toronjil
1,SYN-ORD-000002,eucalipto|vaporub
2,SYN-ORD-000003,vaporub
3,SYN-ORD-000004,lavanda
4,SYN-ORD-000005,lavanda|toronjil
5,SYN-ORD-000006,romero
6,SYN-ORD-000007,jengibre
7,SYN-ORD-000008,copal|manzanilla
8,SYN-ORD-000009,anis-estrella|manzanilla
9,SYN-ORD-000010,eucalipto


In [4]:
basket_sizes = transactions['items'].str.split('|', regex=False).str.len()
summary = pd.Series({
    'canastas': len(transactions),
    'productos': len({item for value in transactions['items'] for item in value.split('|')}),
    'canastas_multiproducto': int(basket_sizes.ge(2).sum()),
    'porcentaje_multiproducto': round(basket_sizes.ge(2).mean() * 100, 2),
}, name='valor')
display(summary.to_frame())

,valor
canastas,1674.00
productos,16.00
canastas_multiproducto,861.00
porcentaje_multiproducto,51.43


## 3. Reglas aprendidas

- **Soporte:** proporción de todas las canastas que contienen A y B.
- **Confianza:** proporción de canastas con A que también contienen B.
- **Lift:** fuerza de la asociación comparada con la aparición independiente. Un lift mayor que 1 indica asociación positiva.

In [5]:
artifact = json.loads(
    (ARTIFACT_DIR / 'apriori-rules.v1.json').read_text(encoding='utf-8')
)
rules = pd.DataFrame(artifact['rules'])
rules_view = rules.assign(
    antecedente=rules['antecedentNames'].str[0],
    consecuente=rules['consequentName'],
)[['antecedente', 'consecuente', 'support', 'confidence', 'lift', 'cooccurrenceCount']]
display(rules_view.head(12))
display(pd.Series(artifact['metrics'], name='valor').to_frame())

,antecedente,consecuente,support,confidence,lift,cooccurrenceCount
0,Vaporub,Eucalipto,0.035245,0.269406,1.879110,59
1,Eucalipto,Vaporub,0.035245,0.245833,1.879110,59
2,Toronjil,Lavanda,0.031661,0.271795,1.636635,53
3,Vaporub,Menta,0.031661,0.242009,1.731296,53
4,Menta,Vaporub,0.031661,0.226496,1.731296,53
5,Menta,Eucalipto,0.031661,0.226496,1.579808,53
6,Eucalipto,Menta,0.031661,0.220833,1.579808,53
7,Anís Estrella,Manzanilla,0.019713,0.239130,1.429658,33
8,Hierbabuena,Menta,0.021505,0.214286,1.532967,36
9,Lavanda,Toronjil,0.031661,0.190647,1.636635,53


,valor
rules,36.000000
catalogCoverage,1.000000
temporalTop1HitRate,0.149826
temporalTrainTransactions,1339.000000
temporalValidationTransactions,335.000000


## 4. Puerta de calidad

Los controles siguientes comprueban el contrato exacto, la cobertura del catálogo y que las reglas respeten los umbrales declarados.

In [6]:
assert list(transactions.columns) == ['tid', 'items']
assert transactions['tid'].is_unique
assert artifact['training']['transactions'] == len(transactions)
assert artifact['metrics']['catalogCoverage'] == 1.0
assert all(rule['lift'] >= artifact['training']['minLift'] for rule in artifact['rules'])
print('APROBADO: dataset transaccional y reglas Apriori listos para integración.')

APROBADO: dataset transaccional y reglas Apriori listos para integración.
